In [ ]:
%matplotlib qt

import numpy as np
import hyperspy.api as hs
import scipy as spy
import matplotlib as plt
from hyperspy.signals import Signal2D

plt.rcParams.update({'font.size': 20})

In [ ]:
#Load data
Image = hs.load(r"C:\Users\pycbr\Documents\PhD\Paper1\Suplimentary\AlmostAllTEM\Image_0030.dm3")

#Normalise data if required
Image.data = Image.data.astype('float32')
Image.data *= 1 / Image.data.max()

#Rotate if necessary
Image.map(spy.ndimage.rotate, angle=0, reshape=False)
Image.plot()

In [ ]:
#Fourier transform image
FourierTransform = Image.fft(shift = True, apodization="hamming")
FourierTransform.plot(power_spectrum = True, vmin="20th", cmap = 'inferno')

In [ ]:
#Crop the Fourier transformed image into a region "BoxRadius" around the center
Shape = FourierTransform.data.shape
x_center = Shape[0]//2
y_center = Shape[1]//2
BoxRadius = 100

fft_cropped = FourierTransform.isig[x_center-BoxRadius:x_center+BoxRadius, y_center-BoxRadius:y_center+BoxRadius]

fft_cropped.plot(power_spectrum = True,
                 vmin="90th", 
                 cmap = 'inferno')

In [ ]:
Data_Array = np.array(fft_cropped)

In [ ]:
#Calculate the power spectrum and apply a cutoff to clean up image
Power_Spectrum = np.log(np.real(Data_Array*np.conjugate(Data_Array)))

BoolTest = Power_Spectrum > 0.5*Power_Spectrum.max()

Power_Spectrum = BoolTest*Power_Spectrum

In [ ]:
Power_Data = Signal2D(Power_Spectrum)
Power_Data.calibrate(x0=0, y0=0, x1=50, y1=50, new_length=9, units="nm^-1", interactive=False)
Power_Data.plot(norm='linear',
               cmap = 'inferno',
               )

In [ ]:
#Find the location of all the spots in the power spectrum
A = Power_Data.find_peaks(method='minmax', interactive=False)

#Recenter coordinates
Center = Power_Data.data.shape
PeakCoords = A.data[0] - (np.array(Center)//2 +1)
    # method='zaefferer', grad_threshold=0.1, window_size=40, distance_cutoff=50.0)

In [ ]:
#Convert to polar coordinates if needed
NumberOfSpots = len(PeakCoords)
PolarCoords = np.zeros((NumberOfSpots, 2))

i = 0
while i < NumberOfSpots:
    
    PolarCoords[i][0] = 0.12*np.sqrt(PeakCoords[i][0]**2 + PeakCoords[i][1]**2)
    PolarCoords[i][1] = np.arctan(PeakCoords[i][0]/PeakCoords[i][1])
    i += 1 
    
PolarCoords

In [ ]:
Power_Data.plot(norm='log',
               cmap = 'inferno',
               vmin = '60th')